# Hybrid RAG: a simple, step-by-step example

**RAG** (Retrieval-Augmented Generation) = first **search** your documents for the parts that match a question,
then give those parts to an LLM and ask it to answer **from them**.

**Hybrid RAG** = do that search in **two ways at once** and combine the results.

| Search | How it finds chunks | Good at | Weak at |
|---|---|---|---|
| **Keyword search** (BM25) | counts the question's **exact words** in each chunk | exact terms: years (*2019*), codes (*HR-LV-2022*), names | different wording: *"look after a sick parent"* won't match *"Caregiver leave"* |
| **Meaning search** (vectors) | compares the **meaning** of the question and each chunk | paraphrases and synonyms | exact numbers and codes: *2019* and *2020* look almost the same to it |

Each search is strong where the other is weak, so together they find the right chunk more often.

**Our documents:** `Leave_Policy` has **12 leave policies (2015–2026)** from one company. They are almost identical;
only the numbers change from year to year. A question like *"earned leave in 2019?"* matches **all 12** by meaning;
only keyword search can tell 2019 apart. That makes this a good dataset to see hybrid search work.

### Steps
| Step | What happens |
|---|---|
| 1 | Imports and settings |
| 2 | Read the PDFs |
| 3 | Split the text into chunks |
| 4 | Build the **keyword** index (BM25) |
| 5 | Build the **meaning** index (embeddings in Pinecone) |
| 6 | Keyword search function |
| 7 | Meaning search function |
| 8 | **Combine** both results (Reciprocal Rank Fusion) |
| 9 | Compare the three searches side by side |
| 10 | **Corrective RAG:** connect to Gemini, settings, `retrieve` (top 3 keyword + top 3 semantic) |
| 11–12 | Load the **cross-encoder**; **grade** every retrieved chunk with it |
| 13–15 | **Decide** the verdict; **rewrite** the question (up to 3 times); merge results |
| 16–17 | Optional web fallback; answer from the evidence sentences |
| 18–19 | The full Corrective RAG pipeline, and try it |

### Step 1: Imports and settings

**What this cell does:** loads the libraries and sets a few values used later.

| Library | Used for |
|---|---|
| `pypdf` | reading text from the PDFs |
| `RecursiveCharacterTextSplitter` | cutting text into small chunks |
| `BM25Okapi` (rank_bm25) | keyword search |
| `SentenceTransformer` | turning text into vectors for meaning search |
| `Pinecone` | the database that stores the vectors and searches them |
| `genai` | Google Gemini, which writes the final answer |

**Settings**
- `DATA_FOLDER = "../Leave_Policy"`: where the PDFs are. `..` means *one folder up*, because this notebook is inside `Hybrid_RAG/`.
- `NAMESPACE = "hybrid-rag"`: our own section inside the Pinecone index, so we don't mix with other notebooks' data.
- `load_dotenv(find_dotenv(usecwd=True))`: loads the API keys from the `.env` file, searching from this folder upwards.

**Output:** nothing.

In [1]:
import os
import re
import json
import time

from pypdf import PdfReader
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from google import genai
from google.genai import types

load_dotenv(find_dotenv(usecwd=True))

DATA_FOLDER = "../Leave_Policy"
NAMESPACE = "hybrid-rag"

c:\Users\DELL\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Read the PDFs

**What this cell does:** reads every leave-policy PDF, page by page.

1. Opens `manifest.json`, a list of the policies with each one's `year`, `file` name and document id (`doc_id`, e.g. `HR-LV-2019`).
2. For each policy, opens the PDF and takes the text of every page (`page.extract_text()`).
3. Saves each page with its **year** and **page number**, so later we can say where an answer came from.

**Output:** how many pages were read, and the start of the first page.

In [2]:
with open(os.path.join(DATA_FOLDER, "manifest.json"), encoding="utf-8") as f:
    manifest = json.load(f)

pages = []
for doc in manifest:
    reader = PdfReader(os.path.join(DATA_FOLDER, doc["file"]))
    for page_no, page in enumerate(reader.pages, start=1):
        pages.append({
            "year": doc["year"],
            "doc_id": doc["doc_id"],
            "page": page_no,
            "text": page.extract_text(),
        })

print(f"Read {len(pages)} pages from {len(manifest)} PDFs")
print("\nFirst page starts with:\n", pages[0]["text"][:300])

Read 47 pages from 12 PDFs

First page starts with:
 Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources 


### Step 3: Split the text into chunks

**What this cell does:** cuts every page into small pieces (**chunks**) of about 800 characters.

**Why chunks?** Searching works best on small, focused pieces. If a whole page were one piece, a question about one
sentence would compete with everything else on that page.

- `chunk_size=800`: each chunk is at most about 800 characters.
- `chunk_overlap=100`: neighbouring chunks share up to 100 characters, so a sentence cut at the border isn't lost.

**Important trick: a label in front of every chunk.**
```
Leave Policy 2019 (HR-LV-2019), page 2: Employees receive 10 days of Casual Leave ...
```
A chunk from the middle of a page might not mention its year at all. The label makes sure **both** searches always
know which year's policy a chunk belongs to.

Each chunk gets an id like `2019-p2-c1` (= 2019 policy, page 2, chunk 1).
`chunk_by_id` lets us look up a chunk's text from its id later.

**Output:** the number of chunks and one example.

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

chunks = []
for p in pages:
    for i, piece in enumerate(splitter.split_text(p["text"])):
        chunks.append({
            "id": f"{p['year']}-p{p['page']}-c{i}",
            "year": p["year"],
            "page": p["page"],
            "text": f"Leave Policy {p['year']} ({p['doc_id']}), page {p['page']}: {piece}",
        })

chunk_by_id = {c["id"]: c for c in chunks}

print(f"Created {len(chunks)} chunks\n")
print("Example chunk:", chunks[0]["id"])
print(chunks[0]["text"][:400])

Created 147 chunks

Example chunk: 2015-p1-c0
Leave Policy 2015 (HR-LV-2015), page 1: Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources Department
Applies to
All permanent employees of Acme Techno


### Step 4: Build the keyword index (BM25)

**What this cell does:** prepares the keyword search.

**1. `tokenize(text)`: split text into search words**
- `text.lower()`: lowercase, so *Sick* and *sick* match.
- `re.findall(r"[a-z0-9]+", ...)`: keep only letters and digits. `HR-LV-2019` becomes `hr`, `lv`, `2019`.
- Remove **stopwords** (*the, of, what, how, many ...*): they appear everywhere and don't help find the right chunk.

Example: *"How many days of Earned Leave in 2019?"* → `['days', 'earned', 'leave', '2019']`

**2. `BM25Okapi(...)`: the keyword index**

BM25 is the standard keyword-search formula used by search engines. For a question, it gives every chunk a score:
- more of the question's words in the chunk → **higher** score
- **rare** words count more than common ones: *2019* (only in the 2019 policy) counts a lot, *leave* (in every chunk) counts very little
- long chunks don't win just by being long

The same `tokenize` is used for chunks (here) and questions (Step 6), so both are split the same way.

**Output:** the words of an example question.

In [4]:
STOPWORDS = {"a", "an", "the", "of", "in", "on", "for", "to", "and", "or", "is", "are", "was", "were",
             "what", "which", "how", "many", "much", "does", "do", "did", "can", "i", "my", "me",
             "it", "this", "that", "by", "with", "at", "as", "from", "per", "get", "got"}



def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS]
    

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

print(tokenize("How many a days of Earned Leave did employees get in 2019?"))
print(chunks[3])
print(bm25.get_scores(tokenize("How many a days of Earned Leave did employees get in 2019?")))

['days', 'earned', 'leave', 'employees', '2019']
{'id': '2015-p2-c0', 'year': 2015, 'page': 2, 'text': 'Leave Policy 2015 (HR-LV-2015), page 2: Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015\nPage 2\nEmployees receive 8 days of Casual Leave in 2015. It is credited on 1 January and is pro-rated by\ncompleted months for employees joining during the year. Unused Casual Leave lapses on 31 December\n2015 and cannot be carried forward or encashed. Casual Leave of more than 3 days at a stretch is not\npermitted and must be combined with Earned Leave. Half-day leave: Not permitted; minimum leave unit is\n1 day.\n5. Sick Leave (2015)\nEmployees receive 8 days of Sick Leave in 2015, credited on 1 January and pro-rated for new joiners. A\nmedical certificate from a registered medical practitioner is required when Sick Leave exceeds 3 days.\nSick Leave carry forward: Not offered. Sick Leave cannot be encashed.\n6. Earned Leave (2015)'}
[2.92405598 2.69041013 2.15895184 4.394575

### Step 5: Build the meaning index (embeddings in Pinecone)

Meaning search needs every chunk turned into a vector and stored in a vector database. This step is split into
six small cells, **5.1 to 5.6**, one job each.

**What is an embedding?** A list of numbers (here 384) that represents the **meaning** of a text.
Texts with similar meaning get similar numbers, e.g. *"time off to care for a parent"* ≈ *"Caregiver leave"*.

| Cell | Job |
|---|---|
| 5.1 | load the embedding model |
| 5.2 | connect to Pinecone |
| 5.3 | clear old vectors from earlier runs |
| 5.4 | turn every chunk into a vector |
| 5.5 | upload the vectors to Pinecone |
| 5.6 | wait until Pinecone has saved them |

#### 5.1 Load the embedding model

**What this cell does:** loads `all-MiniLM-L6-v2`, the model that turns text into vectors.
- Small and free, and runs **on your computer** (no API key needed).
- Downloaded the first time (about 90 MB), then loaded from the local cache.
- Every text becomes a vector of **384 numbers**.

**Output:** the vector of a short example sentence: its length (384) and its first 5 numbers.

In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

example = model.encode("Caregiver leave")
print("Vector length:", len(example))
print("First 5 numbers:", example[:5])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1959.69it/s]


Vector length: 384
First 5 numbers: [-0.01955347 -0.01959092  0.0015571   0.02956124  0.04665591]


#### 5.2 Connect to Pinecone

**What this cell does:** opens the vector database.
1. Logs in with `PineCone_API_Key` from `.env`.
2. Reads the index name from `Pinecone_Index_Name` (an **index** is Pinecone's name for a table of vectors).
3. Creates the index **only if it doesn't exist yet**:
   - `dimension=384`: must match the model from 5.1
   - `metric="cosine"`: compares vectors by the angle between them (1 = same meaning)
4. `pc.Index(...)` gives us `index`, which we use to upload and search.

**Output:** the index name.

In [6]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=384, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)
print("Connected to Pinecone index:", index_name)

Connected to Pinecone index: agenticai-training


#### 5.3 Clear old vectors

**What this cell does:** deletes all vectors in **our namespace** (`hybrid-rag`) left over from earlier runs, so we start clean.
- Only our namespace is cleared. Other notebooks' data in the same index is not touched.
- The very first time, the namespace doesn't exist yet and Pinecone raises an error; `try / except` simply ignores it.

**Output:** a confirmation line.

In [7]:
try:
    index.delete(delete_all=True, namespace=NAMESPACE)
except Exception:
    pass  # namespace doesn't exist yet (first run)

print(f"Namespace '{NAMESPACE}' is empty")

Namespace 'hybrid-rag' is empty


#### 5.4 Turn every chunk into a vector

**What this cell does:** sends the text of **all** chunks to the model in one call (`model.encode`).
The result is a table with one row per chunk and 384 numbers per row.

**Output:** the table's shape, e.g. `(147, 384)` = 147 chunks × 384 numbers.

In [8]:
vectors = model.encode([c["text"] for c in chunks])

print("Shape:", vectors.shape)

Shape: (147, 384)


#### 5.5 Upload the vectors to Pinecone

**What this cell does:** stores every vector in Pinecone.
1. Builds one **record** per chunk:
   - `id`: the chunk id (e.g. `2019-p2-c1`), so search results can be matched back to the chunk text
   - `values`: the 384 numbers (`.tolist()` turns them into a normal Python list)
   - `metadata`: extra information saved with the vector; here the **year**, so we could filter by year later
2. Uploads them in groups of 100 with `upsert` (= insert, or replace if the id already exists).
   Sending in groups keeps each request small.

**Output:** how many records were sent.

In [9]:
records = []
for i, c in enumerate(chunks):
    records.append({
        "id": c["id"],
        "values": vectors[i].tolist(),
        "metadata": {"year": c["year"]},
    })

for start in range(0, len(records), 100):
    index.upsert(vectors=records[start:start + 100], namespace=NAMESPACE)

print(f"Uploaded {len(records)} vectors")

Uploaded 147 vectors


#### 5.6 Wait until Pinecone has saved them

**What this cell does:** Pinecone needs a few seconds before new vectors can be searched. If we searched straight away,
some chunks could be missing from the results.

- `count_vectors()` asks Pinecone how many vectors are stored in our namespace.
- The loop checks every 2 seconds until the count equals the number of chunks (giving up after 60 seconds).

**Output:** how many vectors are stored. It should equal the number of chunks from Step 3.

In [10]:
def count_vectors():
    ns = index.describe_index_stats().namespaces.get(NAMESPACE)
    return ns.vector_count if ns else 0


for _ in range(30):
    if count_vectors() == len(chunks):
        break
    time.sleep(2)

print(f"Stored {count_vectors()} of {len(chunks)} vectors in Pinecone")

Stored 147 of 147 vectors in Pinecone


### Step 6: Keyword search

**What `keyword_search(question)` does:**
1. Splits the question into words with `tokenize` (Step 4).
2. `bm25.get_scores(...)` gives **every** chunk a score for those words.
3. Sorts the chunks from highest to lowest score and returns the top `k` (default 20) as `(chunk_id, score)`.

**Output:** the top 3 keyword results for a test question.

In [11]:
def keyword_search(question, k=20):
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(zip(chunks, scores), key=lambda pair: pair[1], reverse=True)
    return [(c["id"], score) for c, score in ranked[:k]]


for chunk_id, score in keyword_search("earned leave in 2019")[:3]:
    print(f"{score:6.2f}  {chunk_id}")

  6.73  2019-p2-c2
  6.70  2019-p4-c0
  6.55  2019-p1-c0


### Step 7: Meaning search

**What `vector_search(question)` does:**
1. Turns the question into a vector with the **same** model used for the chunks (Step 5).
2. Asks Pinecone for the `k` (default 20) most similar chunk vectors.
3. Returns them as `(chunk_id, score)`. The score is the cosine similarity (closer to 1 = more similar meaning).

**Output:** the top 3 meaning results for the same test question as Step 6. Compare the years.

In [12]:
def vector_search(question, k=20):
    result = index.query(vector=model.encode(question).tolist(), top_k=k, namespace=NAMESPACE)
    return [(match.id, match.score) for match in result.matches]


for chunk_id, score in vector_search("earned leave in 2019")[:3]:
    print(f"{score:6.3f}  {chunk_id}")

 0.681  2019-p2-c2
 0.642  2018-p2-c2
 0.638  2019-p1-c2


### Step 8: Combine both results (Reciprocal Rank Fusion)

This is the **heart of Hybrid RAG**: merge the keyword list and the meaning list into **one** ranking.

**Problem:** the scores can't be added directly. Keyword scores are roughly 0–30, meaning scores 0–1.

**Solution: Reciprocal Rank Fusion (RRF).** Ignore the scores and use only each chunk's **position** (rank) in each list:

```
RRF score = 1 / (60 + rank in keyword list)  +  1 / (60 + rank in meaning list)
```

| Chunk | Keyword rank | Meaning rank | RRF score |
|---|---|---|---|
| A | 1 | 2 | 1/61 + 1/62 = **0.0325** ← in both lists, so it wins |
| B | 2 | not found | 1/62 = 0.0161 |
| C | not found | 1 | 1/61 = 0.0164 |

A chunk that **both** searches like gets two contributions and rises to the top.
`60` is the standard value; it stops a single #1 rank from dominating.

**`hybrid_search(question)`** runs both searches, merges them with RRF, and returns the top 5 chunks.

**Output:** nothing (used in the next steps).

In [13]:
def hybrid_search(question, top_k=5):
    rrf_scores = {}
    for results in [keyword_search(question), vector_search(question)]:
        for rank, (chunk_id, _score) in enumerate(results, start=1):
            rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)

    ranked = sorted(rrf_scores.items(), key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]

### Step 9: Compare keyword vs meaning vs hybrid

**What this cell does:** for each test question, prints the **top 3** results of each search, showing the **year** and
**page** of each chunk, so you can see where each search succeeds or fails.

**What to look for:**

| Question | Keyword search | Meaning search | Hybrid |
|---|---|---|---|
| *earned leave in **2019**?* | all 2019 ✔ | mixes in another year (2018) ✘ | all 2019 ✔ |
| *time off to **look after a sick family member**?* | unrelated chunks: none of these words is in "Caregiver leave" ✘ | family-related leave ✔ | a **caregiver leave** chunk in the top 3 ✔ |
| *casual leave in policy **HR-LV-2022**?* | all 2022 ✔ | no 2022 chunk: the code means nothing to it ✘ | a **2022** chunk on top ✔ |

Each single search fails at least once; hybrid gets a useful result for all three. That's why we combine them.

(Scores can vary slightly between runs; the pattern stays the same.)

In [14]:
def show(chunk_id, score):
    c = chunk_by_id[chunk_id]
    text = c["text"].split(": ", 1)[1][:60].replace("\n", " ")
    print(f"     {score:7.3f}  year {c['year']}  page {c['page']}  | {text}")


def compare(question):
    print("QUESTION:", question)
    print("  Keyword search:")
    for chunk_id, score in keyword_search(question)[:3]:
        show(chunk_id, score)
    print("  Meaning search:")
    for chunk_id, score in vector_search(question)[:3]:
        show(chunk_id, score)
    print("  Hybrid search:")
    for chunk_id, score in hybrid_search(question)[:3]:
        show(chunk_id, score)
    print()


compare("How many days of earned leave did employees get in 2019?")
compare("Can I take time off to look after a sick family member?")
compare("What is the casual leave entitlement in policy HR-LV-2022?")

QUESTION: How many days of earned leave did employees get in 2019?
  Keyword search:
       8.963  year 2019  page 2  | a weekly off or public holiday; must be availed within 60 da
       7.901  year 2019  page 1  | Acme Technologies India Pvt. Ltd. | Leave Policy 2019 | HR-L
       7.762  year 2019  page 4  | Acme Technologies India Pvt. Ltd. | Leave Policy 2019 | HR-L
  Meaning search:
       0.660  year 2019  page 2  | Sick Leave carry forward: Not offered. Sick Leave cannot be 
       0.659  year 2019  page 1  | - Paternity leave: changed from 5 days to 7 days. - Adoption
       0.634  year 2018  page 1  | - Work from home policy: INTRODUCED - Ad-hoc, up to 2 days p
  Hybrid search:
       0.032  year 2019  page 2  | Sick Leave carry forward: Not offered. Sick Leave cannot be 
       0.031  year 2019  page 1  | - Paternity leave: changed from 5 days to 7 days. - Adoption
       0.030  year 2019  page 3  | 11. Applying for and Approving Leave (2019) How to apply: On

QUESTION: Can I

---
## Corrective RAG (CRAG) with a cross-encoder grader

Plain RAG would hand **whatever the search finds** to Gemini. If the search returns the wrong chunks (wrong
year, wrong leave type, nothing useful), Gemini still answers, often wrongly.

**Corrective RAG** adds a **quality check** between searching and answering, and **corrects** course when the results
are not good enough. It reuses everything above: `keyword_search` (Step 6), `vector_search` (Step 7), and `chunk_by_id` (Step 3).

In this notebook the **grading** is done by a local **cross-encoder** model, not by Gemini. Gemini is still used to
**rewrite** the question and to **write** the final answer.

```
 question
    │
    ▼
 1. RETRIEVE  ── top 3 keyword results (BM25) + top 3 meaning results (Pinecone)
    │
    ▼
 2. GRADE     ── cross-encoder scores every chunk: relevant / partial / irrelevant
    │
    ▼
 3. DECIDE    ── CORRECT?   ──yes──────────────────────────────┐
    │                                                          │
    │ AMBIGUOUS or INCORRECT                                   │
    ▼                                                          │
    rewrite the question, search again (up to 3 times) ────────┤
    │                                                          │
    │ still no relevant chunk                                  │
    ▼                                                          ▼
    "answer not found" (or web search, if allowed)          4. REFINE  ── keep only the evidence sentences
                                                               │
                                                               ▼
                                                  5. GENERATE ── answer with citations
```

| Verdict | When | What we do |
|---|---|---|
| **CORRECT** | at least one chunk is **relevant** | answer from the good chunks |
| **AMBIGUOUS** | only **partial** matches (related, but not the answer) | rewrite the question and search again (up to 3 times); if still no relevant chunk: *"answer not found"* |
| **INCORRECT** | **nothing** useful | rewrite the question and search again (up to 3 times); if still no relevant chunk: *"answer not found"* |

### Step 10: Connect to Gemini, settings and helpers

**Settings**

| Setting | Meaning |
|---|---|
| `KEYWORD_TOP_K` | how many chunks we take from **keyword** search (3) |
| `SEMANTIC_TOP_K` | how many chunks we take from **meaning** (semantic) search (3) |
| `MAX_RETRIES` | how many times we may **rewrite the question** when no chunk is relevant (3) |
| `ALLOW_WEB` | `True` = try a Google web search instead of answering *"not found"* |

**Connect to Gemini:** `genai.Client(...)` logs in with `GEMINI_API_KEY` from `.env`.
If you see **503 UNAVAILABLE**, Google's servers are busy: wait a minute and run the cell again,
or change the model name (e.g. to `gemini-3.5-flash-lite`).

**`ask_gemini(prompt, as_json, tools)`** sends a prompt to Gemini and returns the reply text. Options:

| Option | Meaning |
|---|---|
| `temperature=0` | no randomness: the same prompt gives (almost) the same reply |
| `tools=[...]` | lets Gemini use **Google Search** (used for the web fallback in Step 16) |

**`retrieve(question)`** is step **1. RETRIEVE**:
1. Takes the **top 3** chunks from `keyword_search` (Step 6) and the **top 3** from `vector_search` (Step 7).
2. Joins them into one list: keyword results first, then meaning results. A chunk found by **both** searches is
   kept only once, so the list has **up to 6** chunks.
3. Turns each id into a **chunk dictionary** with:
   - `label`: e.g. `Leave Policy 2019 (HR-LV-2019), page 2` (the part of the chunk text before the first `: `)
   - `found_by`: `keyword`, `semantic` or `keyword + semantic`, so you can see which search found it

All of these chunks then go to the **Corrective RAG** grader (Step 12), which decides which ones are actually useful.

**Output:** the chunks found for a test question, and which search found each one.

In [15]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

KEYWORD_TOP_K = 3   # chunks taken from keyword search (BM25)
SEMANTIC_TOP_K = 3  # chunks taken from meaning search (Pinecone)
MAX_RETRIES = 3     # question rewrites when no chunk is relevant
ALLOW_WEB = False   # True = allow web search as a last resort


def ask_gemini(prompt, tools=None):
    config = types.GenerateContentConfig(temperature=0, tools=tools)
    response = client.models.generate_content(model="gemini-3.5-flash", contents=prompt, config=config)
    return response.text or ""


def retrieve(question):
    """Top 3 keyword + top 3 semantic chunks (duplicates removed), as chunk dictionaries."""
    keyword_ids = [chunk_id for chunk_id, _score in keyword_search(question)[:KEYWORD_TOP_K]]
    semantic_ids = [chunk_id for chunk_id, _score in vector_search(question)[:SEMANTIC_TOP_K]]

    found = []
    for chunk_id in dict.fromkeys(keyword_ids + semantic_ids):  # keeps order, drops duplicates
        searches = [name for name, ids in (("keyword", keyword_ids), ("semantic", semantic_ids))
                    if chunk_id in ids]
        c = chunk_by_id[chunk_id]
        found.append({**c,
                      "label": c["text"].split(": ", 1)[0],
                      "found_by": " + ".join(searches)})
    return found


for c in retrieve("How many days of earned leave did employees get in 2019?"):
    print(f"{c['found_by']:<20}  {c['label']}")

keyword               Leave Policy 2019 (HR-LV-2019), page 2
keyword               Leave Policy 2019 (HR-LV-2019), page 1
keyword               Leave Policy 2019 (HR-LV-2019), page 4
semantic              Leave Policy 2019 (HR-LV-2019), page 2
semantic              Leave Policy 2019 (HR-LV-2019), page 1
semantic              Leave Policy 2018 (HR-LV-2018), page 1


### Step 11: Load the cross-encoder

**What is a cross-encoder?** The embedding model from Step 5 is a **bi-encoder**: it turns the question and each chunk
into vectors **separately** and compares them. That is fast (good for searching thousands of chunks) but rough.

A **cross-encoder** reads the question and one chunk **together**, as a pair, and outputs one number: how well this
chunk answers this question. It is slower (one model run per pair), but much more accurate. That makes it ideal for
**checking a handful of retrieved chunks**, which is exactly what grading is.

| | Bi-encoder (Step 5) | Cross-encoder (here) |
|---|---|---|
| Input | question **or** chunk | question **and** chunk, together |
| Output | a vector | a relevance score |
| Used for | searching all chunks | grading the few chunks we found |

**What this cell does:**
1. Loads `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small free model trained on search questions. Like the embedding
   model, it runs **on your computer** (no API key) and is downloaded the first time (about 90 MB).
2. `activation_fn=torch.nn.Sigmoid()` turns the raw score into a number from **0** (no match) to **1** (strong match),
   so we can use simple thresholds in Step 12.

**Output:** the scores of three example pairs: a direct answer, a different leave type, and unrelated text.

In [1]:
import torch
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", activation_fn=torch.nn.Sigmoid())

question = "How many days of earned leave did employees get in 2019?"
examples = ["Earned Leave entitlement in 2019 is 18 days, accruing at a monthly rate.",
            "Marriage leave: 7 days for the employee's own marriage.",
            "The office canteen opens at 8 am."]
for text, score in zip(examples, cross_encoder.predict([(question, t) for t in examples])):
    print(f"{score:.3f}  {text}")

c:\Users\DELL\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1060.10it/s]


1.000  Earned Leave entitlement in 2019 is 18 days, accruing at a monthly rate.
0.291  Marriage leave: 7 days for the employee's own marriage.
0.000  The office canteen opens at 8 am.


### Step 12: Grade the retrieved chunks with the cross-encoder (step 2. GRADE)

This is the **core of Corrective RAG**: before using the chunks, the cross-encoder checks each one.

**Settings**

| Setting | Meaning |
|---|---|
| `RELEVANT_SCORE` | score **≥ 0.5** → `relevant` (the chunk answers the question) |
| `PARTIAL_SCORE` | score **≥ 0.1** → `partial` (related, but probably not the answer). Below that → `irrelevant` |
| `EVIDENCE_SENTENCES` | how many of the best sentences we keep from each useful chunk (3) |

These thresholds are a starting point: watch the scores in the log (Step 19) and adjust them if chunks are graded
too strictly or too loosely.

**What `grade_chunks(question, chunks)` does:**
1. Scores every `(question, chunk)` pair in **one** call to `cross_encoder.predict`.
2. Turns each score into a grade with the thresholds above.
3. **Year check:** the cross-encoder doesn't know that the numbers change every year, so a 2018 chunk can score as high
   as a 2019 chunk. If the question names a year (`years_in`) and the chunk is from a **different** year, a
   `relevant` grade is lowered to `partial`.
4. Finds the **evidence** for every useful chunk with `best_sentences`: splits the chunk into sentences
   (`split_sentences`), scores each sentence against the question with the same cross-encoder, and keeps the top 3,
   in their original order. Step 17 sends only these sentences to Gemini.
5. Returns **copies** of the chunks with `score`, `grade` and `evidence` added (`{**c, ...}` copies a dictionary).

**Output:** the score and grade of each chunk for a test question, plus the evidence of the best chunk.

In [ ]:
RELEVANT_SCORE = 0.5     # score >= 0.5 -> relevant
PARTIAL_SCORE = 0.1      # score >= 0.1 -> partial, below -> irrelevant
EVIDENCE_SENTENCES = 3   # best sentences kept from each useful chunk


def years_in(question):
    """Years named in the question, e.g. {2019}."""
    return {int(y) for y in re.findall(r"\b(?:19|20)\d{2}\b", question)}


def split_sentences(chunk_text):
    """Split a chunk (without its label) into sentences / lines."""
    body = chunk_text.split(": ", 1)[-1]
    parts = re.split(r"\n|(?<=[.!?])\s+", body)
    return [s.strip() for s in parts if len(s.strip()) > 3]


def best_sentences(question, chunk, n=EVIDENCE_SENTENCES):
    """The n sentences of the chunk that best match the question, in their original order."""
    sentences = split_sentences(chunk["text"])
    if not sentences:
        return ""
    scores = cross_encoder.predict([(question, s) for s in sentences])
    top = sorted(range(len(sentences)), key=lambda i: scores[i], reverse=True)[:n]
    return "\n".join(sentences[i] for i in sorted(top))


def grade_chunks(question, chunks):
    """Score every chunk with the cross-encoder. Returns copies with 'score', 'grade' and 'evidence'."""
    if not chunks:
        return []
    scores = cross_encoder.predict([(question, c["text"]) for c in chunks])
    asked_years = years_in(question)

    graded = []
    for c, score in zip(chunks, scores):
        score = float(score)
        if score >= RELEVANT_SCORE:
            grade = "relevant"
        elif score >= PARTIAL_SCORE:
            grade = "partial"
        else:
            grade = "irrelevant"

        if grade == "relevant" and asked_years and c["year"] not in asked_years:
            grade = "partial"  # right topic, wrong year

        evidence = best_sentences(question, c) if grade != "irrelevant" else ""
        graded.append({**c, "score": score, "grade": grade, "evidence": evidence})
    return graded


test_question = "How many days of earned leave did employees get in 2019?"
test_graded = grade_chunks(test_question, retrieve(test_question))
for c in test_graded:
    print(f"{c['score']:.3f}  {c['grade']:<10}  {c['label']}")

best = max(test_graded, key=lambda c: c["score"])
if best["grade"] != "irrelevant":
    print("\nEvidence from", best["label"], ":\n" + best["evidence"])

### Step 13: Decide the verdict (step 3. DECIDE)

**What `decide(graded)` does:** turns the grades into one verdict.

| If ... | Verdict |
|---|---|
| any chunk is **relevant** | `CORRECT` |
| otherwise, any chunk is **partial** | `AMBIGUOUS` |
| otherwise | `INCORRECT` |

**Output:** the verdict for the chunks graded in Step 12, and for three hand-made examples.

In [ ]:
def decide(graded):
    if any(c["grade"] == "relevant" for c in graded):
        return "CORRECT"
    if any(c["grade"] == "partial" for c in graded):
        return "AMBIGUOUS"
    return "INCORRECT"


print("Step 12 chunks:", decide(test_graded))
print(decide([{"grade": "partial"}, {"grade": "relevant"}]))    # CORRECT
print(decide([{"grade": "partial"}, {"grade": "irrelevant"}]))  # AMBIGUOUS
print(decide([{"grade": "irrelevant"}]))                        # INCORRECT

### Step 14: Rewrite the question (the correction)

When the verdict is `AMBIGUOUS` or `INCORRECT`, the user's wording may simply not match the policy's wording, so the
cross-encoder scores every chunk low.
E.g. *"time off when my dad is in hospital"* vs. the policy term *"Caregiver leave"*.

**What `rewrite_question(question, attempt, previous)` does:** asks Gemini for a better **search query**:
- use the formal policy terms for the leave type
- keep any year mentioned
- drop filler words
- be **different** from the earlier queries (`previous`), so a retry doesn't repeat the same failed search

If Gemini returns nothing, the original question is kept.

**Output:** a rewritten version of an informal question.

In [ ]:
def rewrite_question(question, attempt, previous):
    prompt = f"""The search below did not find a good answer in the company leave policy
documents (one PDF per year, 2015-2026, with leave types such as casual, sick, earned,
maternity, paternity, caregiver, wellness, sabbatical and so on).

Rewrite the question as a better search query: use the formal policy terms for the leave
type, keep any year mentioned, and drop filler words. Give a different query from the
earlier attempts. Return only the query text.

Original question: {question}
Earlier queries: {previous}
Attempt: {attempt}"""
    return ask_gemini(prompt).strip().strip('"') or question


q = "can I get days off when my dad is in hospital?"
print("Original :", q)
print("Rewritten:", rewrite_question(q, 1, [q]))

### Step 15: Merge results from several searches

After a rewrite we search again. The new chunks are **added** to the ones we already have (the **pool**), so nothing
good from an earlier search is lost.

**What `merge(kept, new)` does:** if the same chunk was found **twice**, it keeps the copy with the **better** grade
(`relevant` 2 > `partial` 1 > `irrelevant` 0).

**Output:** a hand-made example where chunk `A` is upgraded from partial to relevant.

In [ ]:
def merge(kept, new):
    """Add newly graded chunks, keeping the best grade when a chunk appears twice."""
    order = {"relevant": 2, "partial": 1, "irrelevant": 0}
    by_id = {c["id"]: c for c in kept}
    for c in new:
        if c["id"] not in by_id or order[c["grade"]] > order[by_id[c["id"]]["grade"]]:
            by_id[c["id"]] = c
    return list(by_id.values())


first = [{"id": "A", "grade": "partial"}, {"id": "B", "grade": "irrelevant"}]
second = [{"id": "A", "grade": "relevant"}, {"id": "C", "grade": "partial"}]
print(merge(first, second))

### Step 16: Web search fallback (optional)

**What `web_search_answer(question)` does:** the **last resort** when no chunk is relevant, even after 3
rewrites. It lets Gemini use **Google Search** and asks for a short general answer that clearly says it is
**not the company's policy**.

It is only used when `ALLOW_WEB = True` (Step 10). With `False`, the pipeline answers *"answer not found"* instead, which is
the safer choice for HR questions.

**Output:** nothing (used in Step 18).

In [ ]:
def web_search_answer(question):
    """Last resort: answer from Google Search via Gemini grounding."""
    prompt = f"""The company's own leave policy documents do not cover this question.
Give a short general answer from the web and say clearly that it is general information,
not the company's policy.

Question: {question}"""
    return ask_gemini(prompt, tools=[types.Tool(google_search=types.GoogleSearch())])

### Step 17: Generate the answer (steps 4. REFINE and 5. GENERATE)

**What `generate_answer(question, evidence_chunks)` does:**
1. **Refine:** for each chunk it uses only the **evidence sentences** picked by the cross-encoder (Step 12), not the whole
   chunk. This removes unrelated text that could confuse Gemini. If a chunk has no evidence, its full text is used.
2. Labels each one `[Source 1]`, `[Source 2]` ... with its year and page.
3. Asks Gemini to answer **only** from these sources, using the right **year**, and to cite them.

**Output:** an answer built from the chunks graded in Step 12.

In [ ]:
def generate_answer(question, evidence_chunks):
    context = "\n\n".join(
        f"[Source {n}] {c['label']}:\n{c['evidence'] or c['text']}"
        for n, c in enumerate(evidence_chunks, start=1)
    )
    prompt = f"""Answer the question using only the sources below.
These are leave policies for different years; use the policy of the year the question asks
about. If no year is asked, use the latest year in the sources and say which year.
Cite the sources you used, e.g. [Source 2]. If the sources do not contain the answer, say so.

{context}

Question: {question}"""
    return ask_gemini(prompt)


useful = [c for c in test_graded if c["grade"] != "irrelevant"]
print(generate_answer(test_question, useful))

### Step 18: The full Corrective RAG pipeline

**What `corrective_rag(question)` does:** puts Steps 10–17 together.

**The retry loop** (the original question + up to `MAX_RETRIES` = 3 rewrites, so at most 4 searches):
1. Search with the latest query (`retrieve`: top 3 keyword + top 3 semantic chunks).
2. Grade the chunks with the cross-encoder **against the original question** (`grade_chunks`). The rewritten query is only used for
   *searching*; relevance is always judged against what the user actually asked.
3. Add them to the pool (`merge`) and decide the verdict on the **whole pool** (`decide`).
4. `CORRECT` (a **relevant** chunk was found) → stop. Otherwise, rewrite the question (`rewrite_question`) and loop.

**After the loop:**
- **Still no relevant chunk** after 3 rewrites → the user gets *"Answer not found ..."* (or a web search, only if
  `allow_web=True`). Partial chunks alone are **not** enough: they are related, but don't contain the answer.
- **At least one relevant chunk** → `generate_answer` from the relevant chunks (plus any partial ones as
  background), relevant first, then newest year first.

**Returns** a dictionary: `question`, `verdict`, `source` (`documents`, `web` or `none`), `answer` and `sources`.
With `verbose=True` it prints every search, every cross-encoder score and grade, and every verdict, so you can
watch it correct itself.

**`print_result(result)`** prints the answer and the numbered sources with their grade.

**Output:** nothing (used below).

In [ ]:
def corrective_rag(question, allow_web=ALLOW_WEB, verbose=True):
    def log(msg):
        if verbose:
            print(msg)

    log(f"QUESTION: {question}")
    queries = [question]
    pool = []

    # 1-3. Retrieve, grade, decide; rewrite and retry if needed
    for attempt in range(MAX_RETRIES + 1):
        query = queries[-1]
        log(f"\n  Retrieval #{attempt + 1}  query: {query!r}")
        graded = grade_chunks(question, retrieve(query))
        for c in graded:
            log(f"    {c['score']:.3f}  {c['grade']:<10}  {c['found_by']:<20}  {c['label']}")
        pool = merge(pool, graded)

        verdict = decide(pool)
        log(f"  Verdict: {verdict}")
        if verdict == "CORRECT":
            break
        if attempt < MAX_RETRIES:  # AMBIGUOUS / INCORRECT: rewrite and search again
            queries.append(rewrite_question(question, attempt + 1, queries))
            log(f"  Rewrite {attempt + 1} of {MAX_RETRIES}")

    relevant = [c for c in pool if c["grade"] == "relevant"]
    partial = [c for c in pool if c["grade"] == "partial"]

    # Still no relevant chunk after all rewrites: "answer not found" (or web search if allowed)
    if not relevant:
        if allow_web:
            log("  No relevant chunk after all rewrites -> web search fallback")
            return {"question": question, "verdict": verdict, "source": "web",
                    "answer": web_search_answer(question), "sources": []}
        log(f"  No relevant chunk after {MAX_RETRIES} rewrites -> answer not found")
        return {"question": question, "verdict": verdict, "source": "none",
                "answer": "Answer not found. The Leave Policy documents (2015-2026) do not contain "
                          "the answer to this question.",
                "sources": []}

    # 4. Keep the useful chunks: relevant first, newest year first
    evidence = sorted(relevant + partial,
                      key=lambda c: (c["grade"] != "relevant", -(c["year"] or 0)))

    # 5. Generate the answer from the evidence
    return {"question": question, "verdict": decide(pool), "source": "documents",
            "answer": generate_answer(question, evidence),
            "sources": [f"{c['label']} ({c['grade']})" for c in evidence]}


def print_result(result):
    print("\nANSWER:", result["answer"].strip())
    if result["sources"]:
        print("\nSOURCES:")
        for n, s in enumerate(result["sources"], start=1):
            print(f"  [Source {n}] {s}")
    elif result["source"] == "web":
        print("\n(Answer from web search, not from the company policies.)")
    print("-" * 80)

### Step 19: Try Corrective RAG

Watch the log of each question: the cross-encoder scores and grades, the verdict, and whether the question was
rewritten. If the grades look too strict or too loose, adjust `RELEVANT_SCORE` / `PARTIAL_SCORE` in Step 12.

**Question 1: a clear question with a year.** The first search should already find a relevant 2019 chunk, so the
verdict is **CORRECT** after one retrieval. Expected: **18 days**.

In [ ]:
print_result(corrective_rag("How many days of earned leave did employees get in 2019?"))

**Question 2: informal wording.** The policy calls it *Caregiver leave*. The cross-encoder scores informal wording low
(**AMBIGUOUS** or **INCORRECT**), so the question is rewritten with the policy term and searched again, up to 3 times.

In [ ]:
print_result(corrective_rag("Can I get days off when my dad is in hospital?"))

**Question 3: a question the policies can't answer.** The leave policies don't cover salaries. Plain RAG would
still send the closest chunks to Gemini. The cross-encoder scores them all close to 0, so Corrective RAG grades them all
**INCORRECT**, rewrites the question 3 times, and then says *"Answer not found"* instead of risking an invented answer.
This is the main benefit of Corrective RAG.

In [ ]:
print_result(corrective_rag("What is the salary of a senior software engineer?"))

**Question 4: the same question with the web fallback.** With `allow_web=True`, a question the policies can't answer
gets a general answer from Google Search, clearly marked as **not** the company's policy.

In [ ]:
print_result(corrective_rag("What is the salary of a senior software engineer?", allow_web=True))

**Your turn:** ask your own question. Set `verbose=False` to hide the log and see only the answer.

In [ ]:
print_result(corrective_rag("When was fertility treatment leave introduced and how many days does it give?"))